# Prepare MACE Training Dataset from VASP Runs

**Place this notebook in the parent folder** that contains your system directories (e.g. `UiO-66+N2/`, `UiO-66+CH4/`, …).  
Each system folder contains numbered run files `vasprun_*_1.xml`, `vasprun_*_2.xml`, …

**Burn-in rule:**
- `vasprun_*_1.xml` → discard first `BURNIN` steps (equilibration), then stride
- `vasprun_*_2.xml`, `_3.xml`, … → stride only (already equilibrated)

```
📂 parent/                         ← notebook lives here
├── 📂 UiO-66+N2/
│   ├── vasprun_UiO66_N2_1.xml     ← burn-in + stride
│   ├── vasprun_UiO66_N2_2.xml     ← stride only
│   └── vasprun_UiO66_N2_3.xml     ← stride only
├── 📂 UiO-66+CH4/
│   ├── vasprun_UiO66_CH4_1.xml
│   └── ...
└── ...
```

In [ ]:
# ── dependencies ─────────────────────────────────────────────────────────────
# pip install ase tqdm pandas matplotlib
import glob
import os
import re
from pathlib import Path
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from ase.io import read, write

## 1 · Settings

In [ ]:
ROOT        = Path(".")        # parent folder — keep as '.' if notebook is already there
BURNIN      = 2000             # ionic steps to discard at the start of each run
STRIDE      = 10               # keep every Nth frame after burn-in
OUTPUT_XYZ  = "train.xyz"      # combined output file for MACE

## 2 · Discover all `vasprun.xml` files

In [ ]:
all_vaspruns = sorted(ROOT.glob("**/vasprun_*.xml"))

def run_number(path: Path) -> int:
    """Extract the trailing number from vasprun_*_{num}.xml. Returns 0 if not found."""
    m = re.search(r'_(\d+)\.xml$', path.name)
    return int(m.group(1)) if m else 0

def is_first_run(path: Path) -> bool:
    """True only for vasprun_*_1.xml — the file that needs burn-in."""
    return run_number(path) == 1

# Group by top-level system folder, sorted by run number within each system
by_system = defaultdict(list)
for p in all_vaspruns:
    system = p.relative_to(ROOT).parts[0]   # e.g. 'UiO-66+N2'
    by_system[system].append(p)
for sys_name in by_system:
    by_system[sys_name].sort(key=run_number)

print(f"Found {len(all_vaspruns)} vasprun_*.xml across {len(by_system)} system(s):\n")
for sys_name, paths in sorted(by_system.items()):
    print(f"  {sys_name}")
    for p in paths:
        tag = '← burn-in + stride' if is_first_run(p) else '   stride only'
        print(f"    {p.name:40s}  {tag}")

## 3 · Load, burn-in, and stride

In [ ]:
def load_vasprun(path: Path, burnin: int, stride: int):
    """Read one vasprun_*.xml, apply burn-in (only for run #1) + stride."""
    frames = read(str(path), index=":", format="vasp-xml")
    start  = burnin if is_first_run(path) else 0   # ← key logic
    selected = frames[start::stride]
    # Drop frames without forces (e.g. a stray static SCF at the end)
    good = []
    for f in selected:
        try:
            f.get_forces()
            good.append(f)
        except Exception:
            pass
    return frames, good


records     = []   # for the summary table
all_frames  = []   # accumulated frames for writing

for sys_name, paths in sorted(by_system.items()):
    print(f"\n{'─'*60}")
    print(f"  {sys_name}")
    print(f"{'─'*60}")

    sys_kept = 0
    for p in tqdm(paths, desc=sys_name, leave=False):
        applied_burnin = BURNIN if is_first_run(p) else 0
        mode_label     = f'burn-in {applied_burnin} + stride {STRIDE}' if is_first_run(p) else f'stride {STRIDE} only'
        try:
            raw, kept = load_vasprun(p, BURNIN, STRIDE)
            status = "ok"
        except Exception as e:
            raw, kept = [], []
            status = str(e)[:60]

        n_raw  = len(raw)
        n_kept = len(kept)
        sys_kept += n_kept
        all_frames.extend(kept)

        records.append(dict(
            system        = sys_name,
            file          = p.name,
            n_raw         = n_raw,
            burnin_applied= applied_burnin,
            n_kept        = n_kept,
            status        = status,
        ))
        print(f"    {p.name:45s}  {n_raw:5d} steps  [{mode_label}]  → {n_kept:4d} kept")

    print(f"  ► {sys_name} total kept: {sys_kept}")

print(f"\n{'='*60}")
print(f"  Grand total frames: {len(all_frames)}")
print(f"{'='*60}")

## 4 · Summary table

In [ ]:
df = pd.DataFrame(records)

# Per-system subtotals
summary = (
    df.groupby("system")[["n_raw", "n_kept"]]
    .sum()
    .assign(pct_kept=lambda x: (x.n_kept / x.n_raw * 100).round(1))
    .rename(columns={
        "n_raw": "Total steps",
        "n_kept": f"Kept (stride={STRIDE})",
        "pct_kept": "% kept"
    })
)

display(summary)
display(df.style.hide(axis="index"))

## 5 · Frame count bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(max(6, len(by_system)*1.4), 4))

sys_names  = summary.index.tolist()
kept_vals  = summary[f"Kept (stride={STRIDE})"].tolist()
colors     = plt.cm.tab10.colors

bars = ax.bar(sys_names, kept_vals,
              color=[colors[i % len(colors)] for i in range(len(sys_names))],
              edgecolor="white", linewidth=0.8)

ax.bar_label(bars, fmt="%d", padding=4, fontsize=10)
ax.set_ylabel("Frames kept")
ax.set_title(f"Frames per system  (burn-in={BURNIN}, stride={STRIDE})")
ax.set_xticks(range(len(sys_names)))
ax.set_xticklabels(sys_names, rotation=25, ha="right")
ax.set_ylim(0, max(kept_vals) * 1.15)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("frame_counts.png", dpi=150)
plt.show()
print(f"Total: {sum(kept_vals)} frames")

## 6 · Sanity-check one frame

In [ ]:
f = all_frames[0]

print("Formula  :", f.get_chemical_formula())
print("Cell (Å) :\n", f.get_cell())
print("Energy   :", f.get_potential_energy(), "eV")
print("Max force:", f.get_forces().max(), "eV/Å")
try:
    s = f.get_stress()
    print("Stress   :", s)
except Exception:
    print("Stress   : not available")

## 7 · Energy distribution per system

In [ ]:
# Tag each frame with its system name (stored in atoms.info so MACE can filter if needed)
system_frames = defaultdict(list)

idx = 0
for sys_name, paths in sorted(by_system.items()):
    for p in paths:
        row = df[(df.system == sys_name) & (df.file == p.name)]
        n = int(row.n_kept.values[0]) if len(row) else 0
        for f in all_frames[idx:idx+n]:
            f.info["system"] = sys_name
            system_frames[sys_name].append(f)
        idx += n

fig, ax = plt.subplots(figsize=(8, 4))
for i, (sys_name, frames) in enumerate(sorted(system_frames.items())):
    energies = [f.get_potential_energy() / len(f) for f in frames]  # eV/atom
    ax.hist(energies, bins=40, alpha=0.6, label=sys_name, color=colors[i % len(colors)])

ax.set_xlabel("Energy per atom (eV)")
ax.set_ylabel("Count")
ax.set_title("Energy distribution per system")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("energy_distribution.png", dpi=150)
plt.show()

## 8 · Write `train.xyz`

In [ ]:
if not system_frames:
    raise RuntimeError("No frames collected — check your paths and burn-in settings.")

print(f"Writing one {OUTPUT_XYZ} per system folder:\n")
for sys_name, frames in sorted(system_frames.items()):
    if not frames:
        print(f"  ⚠  {sys_name}: no frames, skipping")
        continue
    out_path = ROOT / sys_name / OUTPUT_XYZ
    write(str(out_path), frames, format="extxyz")
    size_mb = os.path.getsize(out_path) / 1024**2
    print(f"  ✓  {sys_name}/{OUTPUT_XYZ}  —  {len(frames)} frames  ({size_mb:.1f} MB)")

## 9 · MACE training command

Run this from the same parent folder after generating `train.xyz`:

```bash
python mace/scripts/run_train.py \
  --train_file train.xyz \
  --valid_fraction 0.1 \
  --model MACE \
  --name uio66_model \
  --energy_weight 1.0 \
  --forces_weight 10.0 \
  --stress_weight 1.0
```

> **Tip:** if your systems have very different energy scales (e.g. N₂ vs CH₄ loading changes the total energy significantly), consider using `--E0s` to subtract per-atom reference energies before training.

In [ ]:
from ase.io import read, write
# Read POSCAR
atoms = read("./UiO-66+CH4/POSCAR.txt")  # or "POSCAR_N2", etc.

# Write LAMMPS data file
write("./UiO-66+CH4/uio66_mace.data", atoms, format="lammps-data", atom_style="atomic")

In [ ]:
import os
from ase.io import read
import numpy as np

UIO66_BASE = {"Zr": 6, "O": 32, "H": 28, "C": 48}

BASE_TYPES = {
    1: (91.224, "Zr", "Zr"),
    2: (15.999, "O",  "O1"),
    3: (15.999, "O",  "O2"),
    4: (15.999, "O",  "O3"),
    5: (1.008,  "H",  "H1"),
    6: (1.008,  "H",  "H2"),
    7: (12.011, "C",  "C1"),
    8: (12.011, "C",  "C2"),
    9: (12.011, "C",  "C3"),
}

# int  → reuse existing framework type (element already in MACE model)
# tuple → new type needed (element not in framework, e.g. N)
GUEST_TYPE_MAP = {
    "N2":  {"N": (10, 14.007, "N")},   # N is new → new type 10
    "H2O": {"O": 2,  "H": 5},          # reuse O1, H1
    "CH4": {"C": 7,  "H": 5},          # reuse C1, H1
    "CO2": {"O": 2,  "C": 7},          # reuse O1, C1
}

TYPE_CYCLE   = {"O": [2,3,4], "H": [5,6], "C": [7,8,9]}
ELEM_TO_TYPE = {"Zr": 1, "O": 2, "H": 5, "C": 7}
PAIR_COEFF_BASE = "Zr O O O H H C C C"

def triclinic(cell):
    a, b, c = cell
    ax = np.linalg.norm(a)
    bx = np.dot(a, b) / ax
    by = np.sqrt(np.dot(b,b) - bx**2)
    cx = np.dot(a, c) / ax
    cy = (np.dot(b,c) - bx*cx) / by
    cz = np.sqrt(np.dot(c,c) - cx**2 - cy**2)
    return ax, by, cz, bx, cx, cy

def frac_to_lammps(s1, s2, s3, ax, by, cz, xy, xz, yz):
    return (ax*s1 + xy*s2 + xz*s3,
                    by*s2 + yz*s3,
                            cz*s3)

def convert(poscar_path, guest, out_path):
    atoms     = read(poscar_path)
    guest_map = GUEST_TYPE_MAP[guest]
    symbols   = atoms.get_chemical_symbols()
    fracs     = atoms.get_scaled_positions()
    ax, by, cz, xy, xz, yz = triclinic(atoms.get_cell())

    # Build type registry — only add truly new types (tuples)
    types_used = dict(BASE_TYPES)
    for elem, val in guest_map.items():
        if isinstance(val, tuple):
            tid, mass, label = val
            types_used[tid] = (mass, elem, label)

    # Split framework vs guest by counting excess over UIO66_BASE
    elem_seen     = {}
    framework_idx = []
    guest_idx     = []
    for i, sym in enumerate(symbols):
        n = elem_seen.get(sym, 0)
        elem_seen[sym] = n + 1
        if n < UIO66_BASE.get(sym, 0):
            framework_idx.append(i)
        else:
            guest_idx.append(i)

    # Framework atoms — typed with cycling
    elem_counter = {}
    framework = []
    for i in framework_idx:
        sym = symbols[i]
        x, y, z = frac_to_lammps(*fracs[i], ax, by, cz, xy, xz, yz)
        if sym in TYPE_CYCLE:
            n   = elem_counter.get(sym, 0)
            tid = TYPE_CYCLE[sym][n % len(TYPE_CYCLE[sym])]
            elem_counter[sym] = n + 1
        else:
            tid = ELEM_TO_TYPE.get(sym, 1)
        framework.append((tid, x, y, z))

    # Guest atoms — reuse int type or use new tuple type
    guest_atoms = []
    for i in guest_idx:
        sym = symbols[i]
        x, y, z = frac_to_lammps(*fracs[i], ax, by, cz, xy, xz, yz)
        val = guest_map[sym]
        tid = val[0] if isinstance(val, tuple) else val
        guest_atoms.append((tid, x, y, z))

    all_atoms = framework + guest_atoms   # framework first, guests last
    n_types   = max(types_used)

    with open(out_path, "w") as f:
        f.write(" # generated by poscar_to_lammps\n\n")
        f.write(f"           {len(atoms)} atoms\n")
        f.write(f"           {n_types} atom types\n\n")
        f.write(f"    0.0000000000    {ax:.10f} xlo xhi\n")
        f.write(f"    0.0000000000    {by:.10f} ylo yhi\n")
        f.write(f"    0.0000000000    {cz:.10f} zlo zhi\n")
        f.write(f"    {xy:.10f}     {xz:.10f}     {yz:.10f} xy xz yz\n")
        f.write("Masses\n\n")
        for tid in sorted(types_used):
            mass, elem, label = types_used[tid]
            f.write(f"           {tid}     {mass:.8f} # {label}\n")
        f.write("\nAtoms # atomic\n\n")
        for i, (tid, x, y, z) in enumerate(all_atoms, 1):
            f.write(f"         {i:4d}     {tid}     {x:.10f}     {y:.10f}     {z:.10f}\n")

    # pair_coeff: only append element labels for genuinely new types
    new_elems = " ".join(
        val[2] for val in guest_map.values() if isinstance(val, tuple)
    )
    pair_coeff = f"{PAIR_COEFF_BASE} {new_elems}".strip()
    print(f"✅  {out_path}  ({len(atoms)} atoms, {n_types} types)")
    print(f"    pair_coeff * * MACE_modelUiO_stagetwo.model-lammps.pt {pair_coeff}\n")

# ── Main loop ─────────────────────────────────────────────────────────────────
folders = {
    "./UiO-66+N2":  "N2",
    "./UiO-66+CH4": "CH4",
    "./UiO-66+H2O": "H2O",
}
for folder, guest in folders.items():
    poscar  = os.path.join(folder, "POSCAR.txt")
    outfile = os.path.join(folder, "uio66_mace.data")
    convert(poscar, guest, outfile)

In [ ]:
import torch
from mace.tools import torch_tools
from mace.tools.compile import prepare
from mace import modules

torch_tools.set_default_dtype("float64")

ckpt = torch.load(
    "./UiO-66+N2/MACE_modelUiO_run-1268_epoch-999_swa.pt",
    map_location="cpu"
)

# The .model file IS the full model object — load that instead
# and just transfer the SWA weights into it
base_model = torch.jit.load(
    "./UiO-66+N2/MACE_modelUiO_stagetwo.model",
    map_location="cpu"
)
print("Base model loaded OK")
print("float_dtype before:", base_model.float_dtype)

In [ ]:
import torch
from mace.tools import torch_tools
from mace.tools.compile import prepare

torch_tools.set_default_dtype("float64")

# These 8.2MB .model files are raw torch objects, NOT jit
model = torch.load(
    "./UiO-66+N2/MACE_modelUiO_stagetwo.model",
    map_location="cpu"
)
print(type(model))
print(type(model).__mro__)